In [1]:
import cv2
import torch
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from collections import defaultdict, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image


sys.path.insert(0, str(Path.cwd() / "MAIN_MODULE" / "src"))
sys.path.insert(0, str(Path.cwd() / "DEPARTMENT_CLASSIFICATION" / "train_model"))
sys.path.insert(0, str(Path.cwd() / "IMG PREPROCESSING"))
sys.path.insert(0, str(Path.cwd() / "CROP_QUALITY_CLASSIFICATION"))

from quality_classifier.predict import quality_classifier
from color_classifier import process_dataset
from crop_extraction import CropCandidate, CropScorer
from predict_single import DepartmentPredictor



sys.path.insert(0, str(Path.cwd() / "VLM_MODULE"))
from detect import load_vlm_model, vlm_predict_crops
sys.path.insert(0, str(Path.cwd() / "LLMTEXT"))
from LLMTEXT.hf_sku_matcher import HFSKUMatcher
from product_matcher import find_top5_matches

from VRAM_CLEAN.vram_cleanup import cleanup_vram, log_vram_usage, deep_cleanup

c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# === МОНИТОРИНГ VRAM ===
from VRAM_CLEAN.vram_cleanup import get_vram_info, cleanup_vram

# Показать информацию о памяти
get_vram_info()
log_vram_usage("initial")


ИНФОРМАЦИЯ О VRAM
GPU: NVIDIA GeForce RTX 5070
  Текущая выделенная память: 0.00 GB
  Текущая зарезервированная память: 0.00 GB
  Максимальная выделенная (за сессию): 0.00 GB
VRAM [initial]: allocated=0.00GB, reserved=0.00GB


In [3]:
config = OmegaConf.load('params.yaml')
root = Path.cwd()
video_folder = root / config.main_extraction.input_folder
yolo_path = root / config.main_extraction.model_path
dept_model_path = root / config.department_classifier.model_path
class_names_path = root / "DEPARTMENT_CLASSIFICATION/train_model/models/class_names.json"

In [4]:
video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".webm"}
videos = sorted(p for p in video_folder.iterdir()
                if p.suffix.lower() in video_extensions and not p.name.startswith("~"))
video_path = videos[0]
print(f"Видео: {video_path.name}")

Видео: 25_12-20.mp4


In [5]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.00GB, reserved=0.00GB


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(str(yolo_path)).to(device)
crop_scorer = CropScorer(
    config.main_extraction.min_crop_width,
    config.main_extraction.min_crop_height,
    config.main_extraction.sharpness_threshold,
)

def _predict_np(self, img, top_k=3):
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = self.transform(image=img)
    x = t['image'].unsqueeze(0).to(self.device)
    with torch.no_grad():
        p = torch.softmax(self.model(x), dim=1)[0]
    top = torch.topk(p, top_k)
    return [(self.class_names[i.item()], v.item() * 100) for v, i in zip(top.values, top.indices)]

DepartmentPredictor.predict_np = _predict_np
classifier = DepartmentPredictor(str(dept_model_path), str(class_names_path))
log_vram_usage("after model loading")

Создана efficientnet-b0 со случайными весами
Модель загружена: efficientnet-b0
Классов: 15
Устройство: cuda
VRAM [after model loading]: allocated=0.05GB, reserved=0.11GB


In [7]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.09GB


In [8]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"{total} frames, {fps:.2f} FPS")

865 frames, 19.96 FPS


In [9]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.09GB


In [10]:
seg_size = total // 5
half_window = 15
step = 2
segment_frames = []
for i in range(5):
    mid = i * seg_size + seg_size // 2
    start = max(0, mid - half_window)
    end = min(total - 1, mid + half_window)
    segment_frames.append(list(range(start, end + 1, step)))
for i, frames in enumerate(segment_frames):
    print(f"  Сегмент {i+1}: {len(frames)} кадров ({frames[0]/fps:.1f}с - {frames[-1]/fps:.1f}с)")

  Сегмент 1: 16 кадров (3.6с - 5.1с)
  Сегмент 2: 16 кадров (12.2с - 13.7с)
  Сегмент 3: 16 кадров (20.9с - 22.4с)
  Сегмент 4: 16 кадров (29.6с - 31.1с)
  Сегмент 5: 16 кадров (38.2с - 39.7с)


In [11]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.09GB


In [12]:
best: dict[int, list[CropCandidate]] = defaultdict(list)
frame_to_seg = {}
for sid, frames in enumerate(segment_frames):
    for f in frames:
        frame_to_seg[f] = sid
seg_preds = [[] for _ in range(5)]

cap = cv2.VideoCapture(str(video_path))
fi = -1
while True:
    ok, fr = cap.read()
    if not ok:
        break
    fi += 1

    if config.main_extraction.rotate_frames:
        fr = cv2.rotate(fr, cv2.ROTATE_90_COUNTERCLOCKWISE)

    res = yolo.track(source=fr, persist=True, tracker=config.main_extraction.tracker_config,
                     conf=config.main_extraction.conf_threshold, iou=config.main_extraction.iou_threshold, verbose=False)[0]

    boxes = None
    if res.boxes is not None and res.boxes.id is not None:
        boxes = res.boxes.xyxy.cpu().numpy()
        for box, conf, tid in zip(boxes, res.boxes.conf.cpu().numpy(), res.boxes.id.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(int, box)
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(fr.shape[1], x2), min(fr.shape[0], y2)
            crop = fr[y1:y2, x1:x2]
            score = crop_scorer.compute_score(crop, conf)
            if score is None:
                continue
            c = CropCandidate(score, crop.copy(), fi, float(conf), [x1, y1, x2, y2])
            best[tid].append(c)
            best[tid].sort(key=lambda x: x.score, reverse=True)
            best[tid] = best[tid][:config.main_extraction.top_k]

    if fi in frame_to_seg:
        sid = frame_to_seg[fi]
        dept, prob = classifier.predict_np(fr)[0]
        seg_preds[sid].append({'frame': fi, 'time': fi / fps, 'department': dept, 'prob': prob})

    if fi % 500 == 0:
        print(f"Frame {fi}/{total}, tracks: {len(best)}")

cap.release()
print(f"Done. Tracks: {len(best)}, crops: {sum(len(v) for v in best.values())}")

# Очистка после трекинга - удаляем YOLO
del yolo
cleanup_vram(verbose=False)
log_vram_usage("after YOLO tracking")

Frame 0/865, tracks: 8
Frame 500/865, tracks: 42
Done. Tracks: 68, crops: 68
VRAM [after YOLO tracking]: allocated=0.05GB, reserved=0.08GB


In [13]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [14]:
# Сбор результатов по предсказанию отдела
rows_seg = []
for sid, preds in enumerate(seg_preds):
    for p in preds:
        rows_seg.append({
            'segment': sid + 1,
            'frame': p['frame'],
            'time_sec': round(p['time'], 1),
            'department': p['department'],
            'probability': round(p['prob'], 1),
        })

department = pd.DataFrame(rows_seg)

In [15]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [16]:
rows_crops = []
for track_id, candidates in best.items():
    for rank, c in enumerate(candidates):
        rows_crops.append({
            'filename': video_path.name,
            'SYS_track_id': track_id,
            'SYS_rank': rank + 1,
            'SYS_score': round(c.score, 1),
            'SYS_confidence': round(c.confidence, 3),
            'product_name': None, 'price_default': None, 'price_card': None,
            'price_discount': None, 'barcode': None, 'discount_amount': None,
            'id_sku': None, 'print_datetime': None, 'code': None,
            'additional_info': None, 'color': None, 'special_symbols': None,
            'frame_timestamp': int(c.frame_index / fps * 1000),
            'x_min': c.bbox[0], 'y_min': c.bbox[1],
            'x_max': c.bbox[2], 'y_max': c.bbox[3],
            'qr_code_barcode': None, 'price1_qr': None, 'price2_qr': None,
            'price3_qr': None, 'price4_qr': None,
            'wholesale_level_1_count': None, 'wholesale_level_1_price': None,
            'wholesale_level_2_count': None, 'wholesale_level_2_price': None,
            'action_price_qr': None, 'action_code_qr': None,
        })

df_crops = pd.DataFrame(rows_crops)

In [17]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [18]:
crops_for_color = [(track_id, candidate.crop) 
                   for track_id, candidates in best.items() 
                   for candidate in candidates]
# Классификация цветов
color_results = process_dataset(crops_for_color)
# Добавление в DataFrame
df_crops['color'] = df_crops['SYS_track_id'].map(color_results)

In [19]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [20]:
# Сбор кропов для классификации качества
crops_for_quality = [(track_id, candidate.crop) 
                     for track_id, candidates in best.items() 
                     for candidate in candidates]

# Классификация качества (мусор/нет)
quality_model_path = root / config.quality_classifier.model_path
trash_map, confidence_map = quality_classifier(str(quality_model_path), crops_for_quality)

# Добавление в DataFrame
df_crops['SYS_trash'] = df_crops['SYS_track_id'].map(trash_map)


Создана MobileNetV3 со случайными весами


In [21]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [22]:
# === ТРАНСФОРМАЦИЯ КООРДИНАТ ===
# Координаты сейчас записаны для кадров, повернутых на 90° против часовой
# Оригинальное видео: 3840x2160 (WxH)
# После поворота: 2160x3840
# Формула преобразования (поворот на 90° по часовой):
# x_min_orig = W_orig - y_max_rot
# y_min_orig = x_min_rot
# x_max_orig = W_orig - y_min_rot
# y_max_orig = x_max_rot

W_orig, H_orig = 3840, 2160

def transform_coords(row):
    x_min_rot = row['x_min']
    y_min_rot = row['y_min']
    x_max_rot = row['x_max']
    y_max_rot = row['y_max']
    
    x_min_orig = W_orig - y_max_rot
    y_min_orig = x_min_rot
    x_max_orig = W_orig - y_min_rot
    y_max_orig = x_max_rot
    
    return pd.Series({
        'x_min_orig': int(x_min_orig),
        'y_min_orig': int(y_min_orig),
        'x_max_orig': int(x_max_orig),
        'y_max_orig': int(y_max_orig)
    })

df_crops[['x_min_orig', 'y_min_orig', 'x_max_orig', 'y_max_orig']] = df_crops.apply(transform_coords, axis=1)

In [23]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [24]:
# === СОЗДАНИЕ crop_array ===
# Создать crop_map из best
crop_map = {}
for tid, candidates in best.items():
    for rank, c in enumerate(candidates):
        crop_map[(tid, rank + 1)] = c.crop

# Создать crop_array
df_crops['crop_array'] = df_crops.apply(
    lambda r: crop_map.get((r['SYS_track_id'], r['SYS_rank']), None), axis=1)


def image_to_vector(arr):
    img = Image.fromarray(arr).convert("RGB")
    img = img.resize((128, 128))
    vec = np.array(img).flatten().astype(np.float32)
    return vec / (np.linalg.norm(vec) + 1e-8)

vectors = np.stack(df_crops["crop_array"].apply(image_to_vector).values)

sim_matrix = cosine_similarity(vectors) 

THRESHOLD = 0.99

upper = np.triu(sim_matrix, k=1)
rows, cols = np.where(upper >= THRESHOLD)

print(f"Найдено дублирующих пар: {len(rows)}\n")
for r, c in zip(rows, cols):
    tid_r = df_crops.iloc[r]["SYS_track_id"]
    tid_c = df_crops.iloc[c]["SYS_track_id"]
    score = sim_matrix[r, c]
    print(f"  [{r}] track_id={tid_r}  <->  [{c}] track_id={tid_c}  |  similarity={score:.6f}")

to_drop = set(cols.tolist())

print(f"\nПод удаление: {len(to_drop)} записей")
print(f"track_id под удаление: {df_crops.iloc[list(to_drop)]['SYS_track_id'].tolist()}")

df_crops = df_crops.drop(index=df_crops.index[list(to_drop)]).reset_index(drop=True)
print(f"\nОсталось: {len(df_crops)} / {len(df_crops) + len(to_drop)}")


Найдено дублирующих пар: 3

  [52] track_id=156  <->  [53] track_id=157  |  similarity=0.996955
  [52] track_id=156  <->  [54] track_id=159  |  similarity=0.995788
  [53] track_id=157  <->  [54] track_id=159  |  similarity=0.998802

Под удаление: 2 записей
track_id под удаление: [157, 159]

Осталось: 66 / 68


In [25]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=0.05GB, reserved=0.08GB


In [26]:
#тест
df_crops = df_crops.head(10)

In [27]:
# === ПОДГОТОВКА ПЕРЕД VLM ===
# Сохраняем системные колонки для восстановления после VLM
sys_columns_to_restore = df_crops[['SYS_track_id', 'SYS_rank', 'SYS_score', 'SYS_confidence']].copy()

# Создаем рабочую копию для VLM
df_crops_for_vlm = df_crops.drop(columns=['SYS_track_id', 'SYS_rank', 'SYS_score', 'SYS_confidence', 'x_min', 'y_min', 'x_max', 'y_max'])

# Переименовываем *_orig колонки в стандартные
df_crops_for_vlm = df_crops_for_vlm.rename(columns={
    'x_min_orig': 'x_min',
    'y_min_orig': 'y_min',
    'x_max_orig': 'x_max',
    'y_max_orig': 'y_max'
})

# crop_array из best (используем оригинальные SYS_track_id и SYS_rank из df_crops)
vlm_model, vlm_processor = load_vlm_model(str(root / 'VLM_MODULE' / 'AVITO'), config_name='High Quality 8-bit Ultra')
crop_map = {}
for tid, candidates in best.items():
    for rank, c in enumerate(candidates):
        crop_map[(tid, rank + 1)] = c.crop

df_crops_for_vlm['crop_array'] = df_crops.apply(
    lambda r: crop_map.get((r['SYS_track_id'], r['SYS_rank']), None), axis=1)

# VLM predict только для не-мусорных строк
df_vlm = vlm_predict_crops(df_crops_for_vlm[df_crops_for_vlm['SYS_trash'] == False], vlm_model, vlm_processor)


W0519 16:45:11.459000 50464 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 729/729 [00:17<00:00, 42.36it/s] 
c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  OCR [5/8]
  OCR [10/8]


In [28]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=7.21GB, reserved=7.30GB


In [29]:

# === ВОССТАНОВЛЕНИЕ ПОСЛЕ VLM ===
# Восстанавливаем системные колонки через merge по индексу
df_vlm = df_vlm.merge(
    sys_columns_to_restore.loc[df_vlm.index],
    left_index=True,
    right_index=True,
    how='left'
)


In [30]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=7.21GB, reserved=7.30GB


In [31]:

df_vlm_match = df_vlm.rename(columns={'product_name': 'ocr_text'})
df_vlm_match = find_top5_matches(df_vlm_match, ocr_col='ocr_text')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10529.32it/s]


In [32]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=7.21GB, reserved=7.30GB


In [33]:
# === Очистка OCR текста от мусора ===
import re

def clean_ocr_text(text: str) -> str:
    """
    Очистка OCR текста от мусора.
    Оставляет только ключевые слова для LLM.
    """
    if pd.isna(text) or text is None:
        return ""
    
    text = str(text)
    
    # Удаление XML-тегов и hex-кодов
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'0x[A-Fa-f]+', ' ', text)
    
    # Удаление специальных символов
    text = text.replace('_', ' ')
    text = text.replace('▁', ' ')
    
    # Оставляем буквы, цифры, %, ., - (для объемов)
    text = re.sub(r'[^\w\sа-яА-Яa-zA-Z.,%-]', ' ', text)
    
    # Удаляем множественные пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Удаляем короткие слова (1-2 символа) - это обычно мусор
    words = [w for w in text.split() if len(w) > 2]
    
    return ' '.join(words).strip()

# Применение очистки к raw_text
df_vlm['raw_text_clean'] = df_vlm['raw_text'].apply(clean_ocr_text)

# Показать примеры очистки
print("=== Примеры очистки OCR текста ===")
for i in range(min(5, len(df_vlm))):
    orig = str(df_vlm.iloc[i]['raw_text'])[:80]
    clean = df_vlm.iloc[i]['raw_text_clean']
    print(f"{i+1}. RAW: {orig}...")
    print(f"   CLEAN: {clean}")
    print()

=== Примеры очистки OCR текста ===
1. RAW: <0x0A>Beluga▁GRILL▁WINE<>295<>295<>295<>4600000123456<>-26%<>12345_678<><>2024-1...
   CLEAN: Beluga GRILL WINE 295 295 295 4600000123456 -26% 12345 678 2024-12-0114

2. RAW: <0x0A>B�ito▁TORO<>659<>529<>21%<>642<><><><>2024-12-0114:30:00...
   CLEAN: ito TORO 659 529 21% 642 2024-12-0114

3. RAW: <0x0A>Бекон▁ТОРО<0x0A>659<0x0A>659<0x0A><0x0A><0x0A>21%<0x0A><0x0A><0x0A>1<0x0A>...
   CLEAN: Бекон ТОРО 659 659 21%

4. RAW: <0x0A>Вино▁ТОРО▁Белое▁сусло▁бел.▁сус▁(Аргентина)▁1л▁Сухое<0x0A>659₽<0x0A>659₽<0x...
   CLEAN: Вино ТОРО Белое сусло бел. сус Аргентина Сухое 659 659 659 4600000123456 -21% 12345 678 2024-12-0114

5. RAW: <0x0A>GUSTARE▁вино▁бел.▁сух.▁(Россия)▁1L<>149<>149<><>4600000123456<>-26%<>12345...
   CLEAN: GUSTARE вино бел. сух. Россия 149 149 4600000123456 -26% 12345 678 2024-12-0114



In [34]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



VRAM: allocated=7.21GB, reserved=7.30GB


In [ ]:
# === Hugging Face LLM для финального выбора SKU ===
# Использует Qwen2.5-7B-Instruct с 4-bit квантованием
# Работает с очищенным текстом (raw_text_clean)
from LLMTEXT.hf_sku_matcher import HFSKUMatcher

# Инициализация matcher (первый запуск скачает модель ~5GB)
hf_matcher = HFSKUMatcher(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    batch_size=8,  # Пакетная обработка
    max_new_tokens=64,
    temperature=0.1,
    log_to_file=False  # Только консоль
)

# Подготовка данных с очищенным текстом
df_vlm_match = df_vlm.rename(columns={'raw_text_clean': 'ocr_text'})
df_vlm_match = find_top5_matches(df_vlm_match, ocr_col='ocr_text')

# Обработка dataframe с топ-5 кандидатами
# LLM будет искать концептуальные совпадения (не точные)
df_final = hf_matcher.process_dataframe(
    df_vlm_match,
    ocr_col='ocr_text',
    output_col='id_sku'
)

# Освобождение VRAM после обработки
hf_matcher.unload_model()

print(f"\n=== РЕЗУЛЬТАТЫ ===")
print(f"Всего строк: {len(df_final)}")
print(f"Найдено SKU: {df_final['id_sku'].notna().sum()}")
print(f"Match rate: {df_final['id_sku'].notna().sum() / len(df_final) * 100:.1f}%")

# Показать несколько примеров
print("\n=== Примеры результатов ===")
for i in range(min(5, len(df_final))):
    row = df_final.iloc[i]
    print(f"\n{i+1}. OCR: {str(row['ocr_text'])[:60]}...")
    print(f"   Top1: {row.get('top1', 'N/A')}")
    print(f"   Top1 SKU: {row.get('top1_sku', 'N/A')}")
    print(f"   LLM выбрал: {row['id_sku']}")
    print(f"   Размышления: {str(row.get('llm_thinking', 'N/A'))[:100]}...")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9800.42it/s]
2026-05-19 16:47:51 [INFO] Загрузка модели: Qwen/Qwen2.5-7B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights:   1%|          | 2/339 [00:00<00:26, 12.76it/s]c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0)

In [ ]:
# === ОЧИСТКА VRAM ===
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
# Проверка памяти
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")



In [ ]:
df_final